# День 2 — обучение GRU в Colab
Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Запускайте ячейки по порядку. Если сессия оборвалась — снова ячейки 1–4, потом ячейку обучения с `--resume`.

## 1. GPU на месте?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 2. Подключить Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!ls -la $W $W/colab_data

## 3. Код в рабочую папку

In [ ]:
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!python -m pytest -q tests/test_day2.py -k 'loss or chunked'

## 4. Данные: parquet с Drive → npy на локальный диск Colab (быстрее, чем читать с Drive)
Займёт ~15–25 минут. Локальный диск очищается при каждом новом подключении.

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid
!python prepare_data.py --src $W/colab_data/train_sub.parquet --out /content/data/train
!df -h /content | tail -1

## 5. Пробный запуск (~5 мин): проверяем скорость и что всё работает

In [ ]:
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid \
    --out /content/smoke --epochs 1 --max-train-seqs 256 --valid-seqs 100

## 6. Основное обучение
Чекпоинты пишутся на Drive после каждой эпохи. Если сессия оборвалась: ячейки 1–4, затем эту же ячейку, добавив `--resume`.

In [ ]:
RUN = W + '/runs/gru128x2_v1'
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid \
    --out $RUN --epochs 6 --batch 64 --chunk 1000

## 7. Экспорт в сабмит (numpy-инференс, проверка совпадения с torch и скорости)

In [ ]:
!python export.py --ckpt $RUN/best.pt --valid-dir /content/data/valid --out $W/submissions/gru128x2_v1
!cat $RUN/history.json | tail -30

## 8. Скачать zip
Файл также лежит на Drive: `wunder/submissions/gru128x2_v1.zip`.

In [ ]:
from google.colab import files
files.download(W + '/submissions/gru128x2_v1.zip')